# DAPRO Walkthrough: Budget Allocation + Calibrated LPB

This notebook gives an end-to-end, minimal demo of the DAPRO pipeline in this repository:

1. Load synthetic survival data.
2. Build model-style predictions (conditional hazards + quantiles).
3. Run DAPRO to allocate budget under an average budget constraint.
4. Calibrate LPB using the known allocation weights.
5. Evaluate coverage/length against target levels.

## 1) Imports and path setup
Run this notebook from `notebooks/` (default), or any location inside the repo.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Make `src/` importable whether we run from repo root or `notebooks/`.
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.dataset_utils.data_utils import get_data
from src.safety_evaluation.survival_utils.conditional_pmf_utils import get_conditional_pmf
from src.safety_evaluation.survival_utils.compute_mean_time_given_pmf import compute_quantile_survival_time
from src.safety_evaluation.utils.utils import split_data
from src.safety_evaluation.budget_allocators.DAPRO import DAPRO
from src.safety_evaluation.calibration.survival_calibration_with_known_weights import SurvivalCalibrationWithKnownWeights
from src.safety_evaluation.construct_calibrated_bound import compute_metrics_bound
from src.train_model.models.utils import SurvivalModelPrediction

torch.set_printoptions(precision=3, sci_mode=False)

## 2) Experiment configuration
These defaults are intentionally small and CPU-friendly.

In [2]:
device = torch.device("cpu")
seed = 7
cal_size = 1000
budget_per_sample = 10.0
tau_prior = 0.56
target_taus = torch.tensor(np.arange(0.05, 0.5, 0.05), dtype=torch.float32)

# LPB code path in this repo uses a dense tau range in log-space.
taus_range = torch.tensor(np.logspace(-3, -0.01, 500), dtype=torch.float32)

# DAPRO controls
projection = "platt"
score = "prob"
n1 = 100

## 3) Load synthetic data and build prediction tensors
For a lightweight demo, we treat synthetic hazards as model predictions.

In [ ]:
(
    p_train, p_cal, p_test,
    x_train, x_cal, x_test,
    y_train, y_cal, y_test,
    t_tilde_train, t_tilde_cal, t_tilde_test,
    e_train, e_cal, e_test,
    b_train, b_cal, b_test,
    n_samples_train, n_samples_cal, n_samples_test,
) = get_data(
    is_real=False,
    device=device,
    dataset_name="synthetic",
    data_setup="default",
    load_x=False,
    seed=seed,
)

# Build a single pool (cal+test) and derive conditional PMFs + quantiles.
probability_est_all = torch.cat([p_cal, p_test], dim=0).float()
t_tilde_all = torch.cat([t_tilde_cal, t_tilde_test], dim=0).long()
conditional_grid_all = get_conditional_pmf(probability_est_all)

# Quantile estimates f(x, tau) used by calibration/allocators.
quantile_est_all = torch.cat([
    compute_quantile_survival_time(
        conditional_grid_all[:, 0].unsqueeze(1),
        quantile=float(tau),
        tail_distribution="geometric",
    )
    for tau in taus_range
], dim=1).float()

quantile_est_all.shape, conditional_grid_all.shape

## 4) Create calibration/test split and run DAPRO budget allocation

In [ ]:
test_size = len(probability_est_all) - cal_size

(
    _x_cal, _x_test,
    t_cal, prob_cal, q_cal,
    t_test, prob_test, q_test,
    cal_idx, test_idx,
) = split_data(
    seed=seed,
    cal_size=cal_size,
    test_size=test_size,
    x_cal_test=None,
    t_tilde_cal_test=t_tilde_all,
    probability_est=probability_est_all,
    quantile_est_cal_test=quantile_est_all,
)

conditional_grid_cal = conditional_grid_all[cal_idx]

cal_pred = SurvivalModelPrediction(quantile_est=q_cal, probability_est=prob_cal)
test_pred = SurvivalModelPrediction(quantile_est=q_test, probability_est=prob_test)

m_upper_bound = int(probability_est_all.shape[1])
allocator = DAPRO(
    conditional_grid=conditional_grid_cal,
    budget_per_sample=budget_per_sample,
    taus_range=taus_range,
    tau_prior=tau_prior,
    m_upper_bound=m_upper_bound,
    projection=projection,
    score=score,
    n1=n1,
)

allocation = allocator.allocate_budget(
    probability_est=cal_pred.probability_est,
    x=None,
    t=t_cal,
    quantile_est=cal_pred.quantile_est,
)

allocation_summary = {
    "allocator": allocator.name,
    "N_cal": int(len(t_cal)),
    "requested_budget_per_sample": budget_per_sample,
    "actual_budget_per_sample": float(allocation.total_budget_used) / len(t_cal),
    "mean_ipcw_weight": float((1.0 / allocation.C_probs).mean()),
    "max_ipcw_weight": float((1.0 / allocation.C_probs).max()),
}
allocation_summary

## 5) Calibrate LPB with DAPRO weights and evaluate coverage/length

In [ ]:
calibration = SurvivalCalibrationWithKnownWeights(
    budget_allocator=allocator,
    taus_range=taus_range,
    tau_prior=tau_prior,
)

calibration.calibrate(
    x_cal=None,
    t_cal=t_cal,
    model_prediction_cal=cal_pred,
)

lpb_test = calibration.get_calibrated_lpb(
    target_taus=target_taus,
    x=None,
    model_prediction=test_pred,
)

coverage, avg_length = compute_metrics_bound(lpb_test, t_test, bound_type="lpb")
target_coverage = 1.0 - target_taus

results_df = pd.DataFrame({
    "tau": target_taus.numpy(),
    "target_coverage": target_coverage.numpy(),
    "empirical_coverage": coverage.numpy(),
    "coverage_gap": (coverage - target_coverage).numpy(),
    "avg_lpb": avg_length.numpy(),
})
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(results_df["tau"], results_df["target_coverage"], marker="o", label="Target")
axes[0].plot(results_df["tau"], results_df["empirical_coverage"], marker="x", label="Empirical")
axes[0].set_title("Coverage vs. target")
axes[0].set_xlabel("tau")
axes[0].set_ylabel("coverage")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].hist(allocation.C.cpu().numpy(), bins=20, alpha=0.8)
axes[1].set_title("DAPRO allocated censoring times (cal set)")
axes[1].set_xlabel("Allocated C")
axes[1].set_ylabel("count")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6) Notes for running on real datasets
- Replace `is_real=False` with `is_real=True` in `get_data(...)`.
- Set `dataset_name` and `data_setup` to one of your prepared real-data folders.
- If you have precomputed model predictions, use `compute_probabilities_and_quantiles(...)` as in `src/safety_evaluation/construct_calibrated_bound.py`.
- Tune `budget_per_sample`, `tau_prior`, `projection`, and `n1`, then compare allocators by repeating the same calibration/evaluation block.